[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/03sarath/adk-agent-engine/blob/main/adk_agentic_ai.ipynb)

# Customer Support AI Agent — Google ADK + Vertex AI Agent Engine

In this notebook, you will build a **production-grade Customer Support Agent** step by step using Google's Agent Development Kit (ADK).

### What you will learn

| Concept | What You Build |
|---|---|
| **Function Tools** | Ticket CRUD operations in Python |
| **Built-in Tools** | Google Search inside an agent |
| **Third-Party Tools** | External CRM API wrapped as a tool |
| **LLM Agent** | Triage agent that answers questions |
| **Multi-Agent System** | Orchestrator routing to specialist agents |
| **Workflow Agent** | Fixed sequential support pipeline |
| **Custom Agent** | Priority classifier in pure Python (no LLM) |
| **Deployment** | Deploy everything to Vertex AI Agent Engine |

### Architecture

```
support_orchestrator  (LLM Agent — routes requests)
├── triage_agent      (LLM Agent — FAQ + Google Search)
├── ticket_agent      (LLM Agent — create / check / update tickets)
└── escalation_agent  (LLM Agent — fetch user info + escalate)
```

## Prerequisites

Before you begin:
1. A Google Cloud project with **Vertex AI API** enabled
2. A **Google Cloud Storage bucket** for staging
3. Run this notebook on **Google Colab**

---
## Section 0: Setup

In [ ]:
# Install required packages
!pip install -q google-adk
!pip install -q google-cloud-aiplatform[adk,agent_engines]>=1.112 google-cloud-storage requests

In [ ]:
# Authenticate with Google Cloud (Colab only)
from google.colab import auth
auth.authenticate_user()

In [ ]:
import vertexai

# UPDATE THESE VALUES
PROJECT_ID = "your-project-id"   # Your GCP project ID
LOCATION   = "us-central1"        # Vertex AI region

# Bucket name is auto-generated from your project ID — no manual setup needed
BUCKET_NAME    = f"{PROJECT_ID}-adk-agent"
STAGING_BUCKET = f"gs://{BUCKET_NAME}"

# For local testing
vertexai.init(
    project=PROJECT_ID,
    location=LOCATION,
    staging_bucket=STAGING_BUCKET,
)

# For deployment
client = vertexai.Client(
    project=PROJECT_ID,
    location=LOCATION,
)

print(f"Project  : {PROJECT_ID}")
print(f"Location : {LOCATION}")
print(f"Bucket   : {STAGING_BUCKET}")

In [ ]:
# Create the GCS bucket automatically — skip if it already exists
!gcloud storage buckets create {STAGING_BUCKET} \
    --project={PROJECT_ID} \
    --location={LOCATION} \
    --uniform-bucket-level-access 2>/dev/null \
    && echo "Bucket created: {STAGING_BUCKET}" \
    || echo "Bucket already exists: {STAGING_BUCKET}"

In [ ]:
# All imports used throughout this notebook
import requests

import vertexai
from vertexai import agent_engines

from google.adk.agents import Agent, SequentialAgent
from google.adk.tools import google_search

print("Imports ready.")

---
## Part 1: Mock Database

We use simple Python dictionaries to simulate a ticket database and a FAQ knowledge base.  
In production, these would be replaced with a real database.

In [ ]:
# In-memory ticket store
TICKETS = {}
_ticket_counter = 1

# FAQ knowledge base — keyword → answer
FAQ = {
    "password": "To reset your password: Settings → Account → Reset Password. Check your email for the link.",
    "billing":  "For billing issues, email billing@support.com or call 1-800-SUPPORT.",
    "refund":   "Refunds are processed within 5–7 business days to your original payment method.",
    "login":    "If you cannot login: clear browser cache, check Caps Lock, or reset your password.",
    "account":  "To update account details: Settings → Profile → Edit.",
    "cancel":   "To cancel your subscription: Settings → Billing → Cancel Plan.",
}

print(f"FAQ has {len(FAQ)} entries.")
print(f"Ticket store is ready (currently {len(TICKETS)} tickets).")

---
## Part 2: Function Tools

**Function Tools** are regular Python functions that an agent can call.  
ADK automatically reads the function name, parameters, and docstring to describe the tool to the LLM.

> Rule: every tool must return a `dict` with a `status` field.

In [ ]:
def get_faq_answer(topic: str) -> dict:
    """Search the FAQ database for an answer to a user question.

    Args:
        topic: The user's question or issue topic.

    Returns:
        dict: status and answer, or a not_found message.
    """
    for keyword, answer in FAQ.items():
        if keyword in topic.lower():
            return {"status": "success", "answer": answer}
    return {"status": "not_found", "message": "No FAQ answer found. Try a web search."}


# Quick test
print(get_faq_answer("how to reset my password"))
print(get_faq_answer("something random"))

In [ ]:
def create_ticket(user_id: str, issue: str, priority: str = "medium") -> dict:
    """Create a new support ticket for the user.

    Args:
        user_id:  The ID of the user submitting the ticket.
        issue:    A short description of the problem.
        priority: Ticket priority — low, medium, or high.

    Returns:
        dict: status and the new ticket_id.
    """
    global _ticket_counter
    ticket_id = f"TKT-{_ticket_counter:04d}"
    _ticket_counter += 1
    TICKETS[ticket_id] = {
        "id": ticket_id,
        "user_id": user_id,
        "issue": issue,
        "priority": priority,
        "status": "open",
        "resolution": None,
    }
    return {"status": "success", "ticket_id": ticket_id, "message": f"Ticket {ticket_id} created."}


def check_ticket_status(ticket_id: str) -> dict:
    """Check the current status of a support ticket.

    Args:
        ticket_id: The ticket ID to look up (e.g. TKT-0001).

    Returns:
        dict: status and full ticket details.
    """
    ticket = TICKETS.get(ticket_id)
    if not ticket:
        return {"status": "error", "message": f"Ticket {ticket_id} not found."}
    return {"status": "success", "ticket": ticket}


def update_ticket(ticket_id: str, status: str, resolution: str = "") -> dict:
    """Update the status and resolution note of a ticket.

    Args:
        ticket_id:  The ticket ID to update.
        status:     New status — open, in_progress, resolved, or closed.
        resolution: Optional note describing how the issue was resolved.

    Returns:
        dict: status and confirmation message.
    """
    if ticket_id not in TICKETS:
        return {"status": "error", "message": f"Ticket {ticket_id} not found."}
    TICKETS[ticket_id]["status"]     = status
    TICKETS[ticket_id]["resolution"] = resolution
    return {"status": "success", "message": f"Ticket {ticket_id} updated to '{status}'."}


# Quick test
result = create_ticket("user_001", "Cannot login to my account", "high")
print(result)
print(check_ticket_status(result["ticket_id"]))

In [ ]:
def escalate_ticket(ticket_id: str, reason: str) -> dict:
    """Escalate a ticket to the human support team.

    Args:
        ticket_id: The ticket ID to escalate.
        reason:    Why this ticket needs human attention.

    Returns:
        dict: status and confirmation message.
    """
    if ticket_id not in TICKETS:
        return {"status": "error", "message": f"Ticket {ticket_id} not found."}
    TICKETS[ticket_id]["status"]            = "escalated"
    TICKETS[ticket_id]["escalation_reason"] = reason
    return {"status": "success", "message": f"Ticket {ticket_id} escalated. A human agent will contact you soon."}


print("All function tools are ready.")

---
## Part 3: Third-Party Tool

**Third-Party Tools** wrap external APIs as ADK tools.  
Here we call the free [JSONPlaceholder](https://jsonplaceholder.typicode.com) API to simulate a CRM system.  
No API key required — perfect for learning.

In [ ]:
def fetch_user_info(user_id: str) -> dict:
    """Fetch user details from the external CRM system.

    Args:
        user_id: The user ID to look up (1–10 for the mock CRM).

    Returns:
        dict: status with name, email, and company — or an error message.
    """
    try:
        url      = f"https://jsonplaceholder.typicode.com/users/{user_id}"
        response = requests.get(url, timeout=5)
        if response.status_code == 200:
            user = response.json()
            return {
                "status":  "success",
                "name":    user["name"],
                "email":   user["email"],
                "company": user["company"]["name"],
            }
        return {"status": "error", "message": "User not found in CRM."}
    except Exception as e:
        return {"status": "error", "message": str(e)}


# Quick test — fetches a real user from the mock API
print(fetch_user_info("1"))

---
## Part 4: LLM Agents

An **LLM Agent** uses a language model to decide which tools to call and in what order.

> **Gemini API Rule:** `google_search` (Built-in Tool) cannot be mixed with Function Tools in the same agent. Keep them in separate agents.

We create two agents:
- `triage_agent`     — uses `get_faq_answer` (Function Tool)
- `web_search_agent` — uses `google_search` (Built-in Tool)

In [ ]:
# Triage Agent — FAQ only (Function Tool)
triage_agent = Agent(
    name="triage_agent",
    model="gemini-2.5-flash",
    description="Answers common customer questions by searching the FAQ database.",
    instruction="""You are a customer support triage agent.
    Use get_faq_answer to search the FAQ for an answer.
    If no answer found, say: 'I could not find an answer in the FAQ.'
    Always be polite and concise.""",
    tools=[get_faq_answer],
)

# Web Search Agent — Google Search only (Built-in Tool)
# Note: google_search cannot be mixed with function tools in the same agent
web_search_agent = Agent(
    name="web_search_agent",
    model="gemini-2.5-flash",
    description="Searches the web to answer questions not found in the FAQ.",
    instruction="""You are a web search agent.
    Use google_search to find a helpful answer for the user's question.
    Always summarize the result in 2-3 clear sentences.""",
    tools=[google_search],
)

print("Triage agent ready   — tool: get_faq_answer (Function Tool)")
print("Web search agent ready — tool: google_search (Built-in Tool)")

---
## Part 5: Multi-Agent System

A **Multi-Agent System** splits work across multiple specialist agents.

```
support_orchestrator
├── triage_agent      ← searches FAQ (Function Tool)
├── web_search_agent  ← searches the web (Built-in Tool)
├── ticket_agent      ← ticket management
└── escalation_agent  ← urgent issues
```

In [ ]:
ticket_agent = Agent(
    name="ticket_agent",
    model="gemini-2.5-flash",
    description="Creates, checks, and updates customer support tickets.",
    instruction="""You are a ticket management agent.
    - Use create_ticket when the user has an unresolved issue.
    - Use check_ticket_status when the user asks about an existing ticket.
    - Use update_ticket when an issue has been resolved.
    Always use user_id="student_001" unless the user explicitly provides a different one.
    Always confirm the ticket ID with the user after creating one.""",
    tools=[create_ticket, check_ticket_status, update_ticket],
)

print("Ticket agent ready.")
print("Tools: create_ticket, check_ticket_status, update_ticket")

In [ ]:
escalation_agent = Agent(
    name="escalation_agent",
    model="gemini-2.5-flash",
    description="Handles urgent issues and escalates to a human support agent.",
    instruction="""You are an escalation specialist.
    Step 1: Use fetch_user_info with user_id="1" to get the user's name and contact details.
    Step 2: Use escalate_ticket to flag the ticket for human review.
    Step 3: Reassure the user that a human agent will contact them soon.
    Never ask the user for a user_id or ticket_id — always use user_id="1" for fetch_user_info.
    Only handle cases where automated resolution is not possible.""",
    tools=[fetch_user_info, escalate_ticket],
)

print("Escalation agent ready.")
print("Tools: fetch_user_info (Third-Party Tool), escalate_ticket (Function Tool)")

In [ ]:
from google.adk.tools.preload_memory_tool import PreloadMemoryTool
from google.adk.agents.callback_context import CallbackContext

# Auto-saves session to Memory Bank after every conversation
# try/except: locally there is no memory service, so we skip silently
async def save_to_memory_callback(callback_context: CallbackContext):
    try:
        await callback_context.add_session_to_memory()
    except Exception:
        pass

root_agent = Agent(
    name="support_orchestrator",
    model="gemini-2.5-flash",
    description="Main customer support agent that routes requests to specialist agents.",
    instruction="""You are the main customer support agent.
    Route each user request to the right specialist:
    - triage_agent     : search the FAQ first for common questions
    - web_search_agent : search the web if FAQ has no answer
    - ticket_agent     : create, check, or update support tickets
    - escalation_agent : urgent issues or highly frustrated users

    You have long-term memory — past facts about this user are loaded automatically.
    Use them to greet the user by name and reference their past issues.

    IMPORTANT — for urgent or escalation requests:
    1. First ask ticket_agent to create a ticket for the issue (use user_id="student_001").
    2. Then pass the returned ticket_id to escalation_agent to escalate it.
    Never ask the user for a ticket ID — create one automatically.

    Always greet the user and confirm which agent you are routing to.""",
    sub_agents=[triage_agent, web_search_agent, ticket_agent, escalation_agent],
    tools=[PreloadMemoryTool()],
    after_agent_callback=save_to_memory_callback,
)

print("Multi-agent system ready.")
print("Orchestrator → [triage_agent, web_search_agent, ticket_agent, escalation_agent]")
print("Memory       → PreloadMemoryTool + after_agent_callback")

---
## Part 6: Workflow Agent

A **Workflow Agent** runs sub-agents in a **fixed order** — no LLM decides the sequence.  
Use this when the steps must always happen in the same order.

| Agent Type | How it decides what to do |
|---|---|
| LLM Agent | The model decides which tool or sub-agent to call |
| SequentialAgent | Always runs sub-agents left to right, in order |

ADK provides three workflow agents:
- `SequentialAgent` — runs sub-agents one after another
- `ParallelAgent`   — runs sub-agents at the same time
- `LoopAgent`       — repeats a sub-agent until a condition is met

> **ADK Rule:** Each agent can only have **one parent**. If an agent is already a sub_agent of the orchestrator, you must create a fresh instance to use it in a workflow.

In [ ]:
# Note: In ADK each agent can only have one parent.
# Agents already belong to support_orchestrator, so we create fresh instances for this workflow.

triage_agent_wf = Agent(
    name="triage_agent_wf",
    model="gemini-2.5-flash",
    description="Answers common customer questions by searching the FAQ database.",
    instruction="""You are a customer support triage agent.
    Use get_faq_answer to search the FAQ for an answer.
    If no answer found, say: 'I could not find an answer in the FAQ.'""",
    tools=[get_faq_answer],
)

ticket_agent_wf = Agent(
    name="ticket_agent_wf",
    model="gemini-2.5-flash",
    description="Creates, checks, and updates customer support tickets.",
    instruction="""You are a ticket management agent.
    - Use create_ticket when the user has an unresolved issue.
    - Use check_ticket_status when the user asks about an existing ticket.
    - Use update_ticket when an issue has been resolved.""",
    tools=[create_ticket, check_ticket_status, update_ticket],
)

# SequentialAgent: always triage first, then ticket — fixed order, no LLM routing
support_workflow = SequentialAgent(
    name="support_workflow",
    description="Fixed support pipeline: triage the issue, then manage the ticket.",
    sub_agents=[triage_agent_wf, ticket_agent_wf],
)

print("Workflow agent ready.")
print("Fixed pipeline: triage_agent_wf → ticket_agent_wf (always in this order)")

---
## Part 7: Custom Agent

A **Custom Agent** has a very tight, specific instruction so it does exactly one thing.  
Instead of a full class, we simply give the agent a strict instruction that replaces the logic.

> Same result as writing Python keyword-matching logic — but just 5 lines of code.

In [ ]:
priority_classifier = Agent(
    name="priority_classifier",
    model="gemini-2.5-flash",
    description="Classifies the priority of a support issue.",
    instruction="""You are a priority classifier. Read the user's message and reply with exactly one of:
    - 'Priority: HIGH'   — if the issue is urgent, broken, critical, or system is down
    - 'Priority: MEDIUM' — if there is a problem, error, or bug but not urgent
    - 'Priority: LOW'    — for general questions or feedback
    Always add one short sentence explaining why.""",
)

print("Custom agent ready — priority_classifier using gemini-2.5-flash.")

---
## Part 8: Test Locally

Before deploying, wrap the agent in `AdkApp` and test it locally.  
We send three different queries to see how the orchestrator routes each one.

In [ ]:
# Wrap root_agent in AdkApp — required for both local testing and deployment
app = agent_engines.AdkApp(
    agent=root_agent,
    enable_tracing=True,
)

print("AdkApp created.")


def extract_text(event: dict) -> str:
    """Pull the final text response out of an event."""
    parts = event.get("content", {}).get("parts", [])
    for part in parts:
        if part.get("text") and not part.get("function_call"):
            return part["text"]
    return ""


async def test_locally():
    session = await app.async_create_session(user_id="student_001")
    print(f"Session created: {session.id}\n")

    queries = [
        "How do I reset my password?",                        # → triage_agent (FAQ)
        "My account is completely broken, this is urgent!",   # → escalation_agent
        "Can you create a ticket for my billing issue?",      # → ticket_agent
    ]

    for query in queries:
        print(f"User : {query}")
        async for event in app.async_stream_query(
            user_id="student_001",
            session_id=session.id,
            message=query,
        ):
            text = extract_text(event)
            if text:
                print(f"Agent: {text}")
        print("-" * 60)


await test_locally()

---
## Part 9: Deploy to Vertex AI Agent Engine

`agent_engines.create()` packages your code, builds a container, and deploys it to Google Cloud.  
This usually takes **3–5 minutes**. You only need to do this once per version.

In [ ]:
print("Deploying to Vertex AI Agent Engine...")
print("This takes 3–5 minutes. Please wait.\n")

remote_app = client.agent_engines.create(
    agent=app,
    config={
        "requirements": [
            "google-adk",
            "google-cloud-aiplatform[adk,agent_engines]",
            "requests",
        ],
        "staging_bucket": STAGING_BUCKET,
    }
)

print("Deployed successfully!")
print(f"Resource name: {remote_app.api_resource.name}")

---
## Part 10: Create a Remote Session

A session keeps conversation history so the agent remembers what was said earlier.  
Each user should have their own session.

In [ ]:
async def create_remote_session():
    session = await remote_app.async_create_session(user_id="student_001")
    print(f"Remote session created: {session['id']}")
    return session


remote_session = await create_remote_session()

---
## Part 11: Query the Remote Agent

Now we send a message to the **deployed** agent running on Vertex AI Agent Engine.  
The agent runs in Google Cloud — not locally on your machine.

In [ ]:
async def query_remote(message: str):
    print(f"User : {message}")
    async for event in remote_app.async_stream_query(
        user_id="student_001",
        session_id=remote_session["id"],
        message=message,
    ):
        text = extract_text(event)
        if text:
            print(f"Agent: {text}")


# Test the remote agent with a real query
await query_remote("My login is completely broken — this is urgent, I need help now!")

---
## Part 12: Memory Bank (Long-Term Memory)

**Memory Bank** stores facts across sessions — the agent remembers users even after they close and reopen the chat.

| Memory Level | Scope |
|---|---|
| Session history | Within one session only |
| **Memory Bank** | **Across ALL sessions — persists forever** |

### How it works

```
Session 1:  "My name is John, billing issue"
            → after_agent_callback auto-saves session to Memory Bank

Session 2 (new session, next day):
            → PreloadMemoryTool auto-loads memories
            → Agent: "Welcome back John! Still having the billing issue?"
```

### Two things to add to the agent
- `PreloadMemoryTool()` — retrieves memories at the start of every turn
- `after_agent_callback` — auto-saves the session to Memory Bank when conversation ends

AdkApp **automatically uses VertexAiMemoryBankService** when deployed on Vertex AI — no extra config needed.

In [ ]:
# Memory is already configured in the agent (Part 5) and deployed (Part 9)
# No extra setup needed here — just run Session 1 and Session 2 below
print("Memory Bank is ready.")
print("PreloadMemoryTool  → auto-retrieves memories at the start of each turn")
print("after_agent_callback → auto-saves session facts when conversation ends")

In [ ]:
# Session 1 — have a conversation
# after_agent_callback automatically saves it to Memory Bank when done
async def session_1():
    session = await remote_app.async_create_session(user_id="student_001")
    print(f"Session 1 started: {session['id']}\n")

    message = "Hi, my name is John and I have a billing issue with my account."
    print(f"User : {message}")
    async for event in remote_app.async_stream_query(
        user_id="student_001",
        session_id=session["id"],
        message=message,
    ):
        text = extract_text(event)
        if text:
            print(f"Agent: {text}")

    print("\nSession ended — after_agent_callback saved facts to Memory Bank automatically.")
    print("Check the Memories tab in Vertex AI console to see: 'John has a billing issue'")


await session_1()

In [ ]:
# Session 2 — brand new session, agent recalls facts from Session 1
# PreloadMemoryTool fires automatically and injects memories into the agent
async def session_2():
    session = await remote_app.async_create_session(user_id="student_001")
    print(f"Session 2 started (new session): {session['id']}\n")
    print("No conversation history — but Memory Bank has facts from Session 1.\n")

    message = "Hi, I'm back. Do you remember my issue?"
    print(f"User : {message}")
    async for event in remote_app.async_stream_query(
        user_id="student_001",
        session_id=session["id"],
        message=message,
    ):
        text = extract_text(event)
        if text:
            print(f"Agent: {text}")


await session_2()

---
## Optional: Clean Up

Delete the deployed agent when you are done to avoid ongoing charges.

In [ ]:
# Uncomment and run this cell to delete the deployed agent
# remote_app.delete(force=True)
# print("Agent deleted.")

---
## Summary

You have built and deployed a production-grade Customer Support Agent using Google ADK.

| What you built | ADK Concept |
|---|---|
| `get_faq_answer`, `create_ticket`, etc. | Function Tools |
| `google_search` inside `web_search_agent` | Built-in Tools |
| `fetch_user_info` calling JSONPlaceholder | Third-Party Tools |
| `triage_agent`, `ticket_agent`, `escalation_agent` | LLM Agents |
| `root_agent` with `sub_agents=[...]` | Multi-Agent System |
| `SequentialAgent([triage_agent, ticket_agent])` | Workflow Agent |
| `priority_classifier` with tight instruction | Custom Agent |
| `agent_engines.create(app, ...)` | Deployment to Vertex AI |
| `VertexAiMemoryBankService` + `add_session_to_memory` | Long-Term Memory |

### Next Steps
- Replace `TICKETS` dict with a real database (Firestore, Cloud SQL)
- Replace `FAQ` dict with a vector search index for semantic matching
- Add authentication to identify real users
- Add a `LoopAgent` to retry failed escalations automatically